In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 1725/1725 [00:00<00:00, 660505.24 examples/s]


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 1725/1725 [00:00<00:00, 31866.56 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=128, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=128, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=128, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([128]),
 'input_ids': torch.Size([128, 100]),
 'token_type_ids': torch.Size([128, 100]),
 'attention_mask': torch.Size([128, 100])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [21]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [22]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [23]:
d_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

w_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
d_model.set_use_module_grafting(False)
d_model.set_use_scc_status(False)

w_model.set_use_module_grafting(False)
w_model.set_use_scc_status(False)

In [25]:
outputs = d_model(**batch)
print(outputs.loss, outputs.logits.shape)

outputs = w_model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.1277, grad_fn=<NllLossBackward0>) torch.Size([128, 2])
tensor(0.6733, grad_fn=<NllLossBackward0>) torch.Size([128, 2])


In [26]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
d_model.to(device)
w_model.to(device)

device

device(type='cuda')

In [28]:
for batch in train_dataloader:
    break
batch = {k: v.to(device) for k, v in batch.items()}
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([128]),
 'input_ids': torch.Size([128, 86]),
 'token_type_ids': torch.Size([128, 86]),
 'attention_mask': torch.Size([128, 86])}

In [29]:
outputs = d_model(**batch)
print(outputs.loss, outputs.logits.shape)

outputs = w_model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.1715, device='cuda:0', grad_fn=<NllLossBackward0>) torch.Size([128, 2])
tensor(0.6695, device='cuda:0', grad_fn=<NllLossBackward0>) torch.Size([128, 2])


# 5. Help Functions

### Width Pruning Functions

In [30]:
def prune_heads_width(model, n_head_prune):
    original_num_params = sum(p.numel() for p in model.parameters())
    heads_to_prune = [prune_head for prune_head in reversed(range(12 - n_head_prune, 12))]
    print("Heads to prune:", heads_to_prune)

    n_layer = len(d_model.bert.encoder.layer)
    for layer in range(n_layer):
        model.prune_heads({layer: heads_to_prune})

    pruned_num_params = sum(p.numel() for p in model.parameters())
    print("Pruning, original num of params:  %.2e, after pruning %.2e (%.1f percents)" % (original_num_params, pruned_num_params, pruned_num_params / original_num_params * 100))

In [31]:
def prune_int_dim(model, int_mask):
    original_num_params = sum(p.numel() for p in model.parameters())
    
    keep_dims_dict = {}
    for layer in range(len(int_mask)):
        keep_dims = []
        for index, value in enumerate(int_mask[layer]):
            if value == 1:
                keep_dims.append(index)
        keep_dims_dict[layer] = keep_dims
    model.prune_int_dims(keep_dims_dict)
    
    pruned_num_params = sum(p.numel() for p in model.parameters())
    
    print("Pruning, original num of params:  %.2e, after pruning %.2e (%.1f percents)" % 
          (original_num_params, pruned_num_params, pruned_num_params / original_num_params * 100))

def get_int_mask(height, width, remove_amt):
    int_mask = torch.ones(height, width)
    start_idx = width - remove_amt  # or int_mask.shape[1] - remove_amt
    int_mask[:, start_idx:] = 0
    return int_mask

# 6 Speed Test

In [32]:
import numpy as np
from time import perf_counter

def time_pipeline(model, batch):
    latencies = []
    
    # Warmup
    for _ in range(10):
        _ = model(**batch)
        
    # Timed run
    for _ in range(100):
        start_time = perf_counter()
        model(**batch)
        latency = perf_counter() - start_time
        latencies.append(latency)
        
    # Compute run statistics
    time_avg_ms = 1000 * np.mean(latencies)
    time_std_ms = 1000 * np.std(latencies)
    print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
    return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

In [33]:
from pathlib import Path

def compute_size(model):
    state_dict = model.state_dict()
    tmp_path = Path("model.pt")
    torch.save(state_dict, tmp_path)
    # Calculate size in megabytes
    size_mb = Path(tmp_path).stat().st_size / (1024 * 1024)
    # Delete temporary file
    tmp_path.unlink()
    print(f"Model size (MB) - {size_mb:.2f}")
    return size_mb

In [34]:
mlp_width = w_model.bert.encoder.layer[0].intermediate.dense.weight.shape[0]
mlp_width

3072

In [35]:
mlp_width_remove = int(mlp_width / 12)
mlp_width_remove

256

In [36]:
def inference_time_on_various_batch_sizes(model, batch, batch_sizes):
    # batch_sizes = [1, 8, 16, 32, 64, 128]
    latencies = []
    
    for batch_size in batch_sizes:
        print("Batch Size:", batch_size)
        modified_batch = {k: v[:batch_size] for k, v in batch.items()}
        latency = time_pipeline(model, batch)
        latencies.append(latency)
    
    return latencies

In [37]:
batch_sizes = [1, 8, 16, 32, 64, 128]

In [38]:
for batch in train_dataloader:
    break
batch = {k: v.to(device) for k, v in batch.items()}
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([128]),
 'input_ids': torch.Size([128, 83]),
 'token_type_ids': torch.Size([128, 83]),
 'attention_mask': torch.Size([128, 83])}

In [39]:
modified_batch = {k: v[:8] for k, v in batch.items()}

In [40]:
{k: v.shape for k, v in modified_batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 83]),
 'token_type_ids': torch.Size([8, 83]),
 'attention_mask': torch.Size([8, 83])}

In [41]:
compute_size(d_model)

Model size (MB) - 417.71


417.7130880355835

In [42]:
original_inference = inference_time_on_various_batch_sizes(d_model, batch, batch_sizes)

Batch Size: 1
Average latency (ms) - 127.65 +\- 0.85
Batch Size: 8
Average latency (ms) - 129.19 +\- 0.52
Batch Size: 16
Average latency (ms) - 130.28 +\- 0.40
Batch Size: 32
Average latency (ms) - 130.64 +\- 0.28
Batch Size: 64
Average latency (ms) - 130.79 +\- 0.43
Batch Size: 128
Average latency (ms) - 130.90 +\- 0.42


### 6 Layer Equivalent

In [43]:
first_6_layers = d_model.bert.encoder.layer[:6]
d_model.bert.encoder.layer = first_6_layers
len(d_model.bert.encoder.layer)

6

In [44]:
int_mask = get_int_mask(12, mlp_width, mlp_width_remove)
int_mask.shape

torch.Size([12, 3072])

In [45]:
sum(int_mask[0])

tensor(2816.)

In [46]:
w_model.bert.encoder.layer[0].output

BertOutput(
  (dense): Linear(in_features=3072, out_features=768, bias=True)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [53]:
prune_int_dim(w_model, int_mask * 6)

Pruning, original num of params:  1.05e+08, after pruning 5.28e+07 (50.4 percents)


/usr/local/lib/python3.10/dist-packages/torch/nn/init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


In [54]:
w_model.bert.encoder.layer[0].output

BertOutput(
  (dense): Linear(in_features=0, out_features=768, bias=True)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [55]:
compute_size(w_model)

Model size (MB) - 201.57


201.572340965271

In [50]:
compute_size(d_model)

Model size (MB) - 255.45


255.4533224105835

In [56]:
original_inference = inference_time_on_various_batch_sizes(w_model, batch, batch_sizes)

Batch Size: 1
Average latency (ms) - 57.03 +\- 0.40
Batch Size: 8
Average latency (ms) - 57.37 +\- 0.22
Batch Size: 16
Average latency (ms) - 57.68 +\- 0.18
Batch Size: 32
Average latency (ms) - 57.96 +\- 0.17
Batch Size: 64
Average latency (ms) - 58.21 +\- 0.17
Batch Size: 128
Average latency (ms) - 58.38 +\- 0.20


In [52]:
original_inference = inference_time_on_various_batch_sizes(d_model, batch, batch_sizes)

Batch Size: 1
Average latency (ms) - 65.70 +\- 0.29
Batch Size: 8
Average latency (ms) - 65.75 +\- 0.30
Batch Size: 16
Average latency (ms) - 65.75 +\- 0.29
Batch Size: 32
Average latency (ms) - 65.73 +\- 0.28
Batch Size: 64
Average latency (ms) - 65.73 +\- 0.29
Batch Size: 128
Average latency (ms) - 65.75 +\- 0.28
